# Finding UIDs
### Lizzie Evans-Ralston

Purpose: Create ID for unique cards - once a card is found as fraudulent, then it always has a isFraud flag. Therefore, the model only needs to be able to predict if the user is likely to be fraudulent, NOT if a transaction itself is fraudulent

## Preprocessing:

In [ ]:
# load data, libraries
import pandas as pd
import numpy as np

df = pd.read_csv('/content/train_transaction.csv')

# join test and train data

df.head()

The columns DX correspond to a time delta from the date the card was created to the transaction date. This means we can hold them constant by subtracting them from TransactionDT. From exploring the metadata, the delta that is most important for determining a UID is D1, which is the delta between the cards creation date and the transaction date.

In [ ]:
# transform DX columns to be constant for each UID despite delta value
# we can determine what is important during feature selection.

df['D1_new'] = df['TransactionDT'] - df['D1'] # only value important for fining UIDs, almost everything else will likely be a subset of this
df['D2_new'] = df['TransactionDT'] - df['D2']
df['D3_new'] = df['TransactionDT'] - df['D3']
df['D4_new'] = df['TransactionDT'] - df['D4']
df['D5_new'] = df['TransactionDT'] - df['D5']
df['D6_new'] = df['TransactionDT'] - df['D6']
df['D7_new'] = df['TransactionDT'] - df['D7']
df['D8_new'] = df['TransactionDT'] - df['D8']
df['D9_new'] = df['TransactionDT'] - df['D9']
df['D10_new'] = df['TransactionDT'] - df['D10']
df['D11_new'] = df['TransactionDT'] - df['D11']
df['D12_new'] = df['TransactionDT'] - df['D12']
df['D13_new'] = df['TransactionDT'] - df['D13']
df['D14_new'] = df['TransactionDT'] - df['D14']
df['D15_new'] = df['TransactionDT'] - df['D15']


D_9 denotes the hour that the transaction occurred instead of a time delta

## Creating the UIDs

We already know certain features will remain constant for a card.
* D1_new: time delta between card use and creation, standardized for any given card
* P_emaildomain: Email domain of card owner
* card1: unique ID for a card
* addr1: address of card owner

Because we know this, we can make a UID based off of these combinations.

In [ ]:
# create base uid based on hypothesis from the metadata

df['uid_1'] = df['D1_new'].astype(str) + '_' + df['P_emaildomain'].astype(str) + '_' + df['card1'].astype(str) + '_' + df['addr1'].astype(str)

We can test how good the UID is by seeing how many UIDs have a both fraud and not fraud flags. Becuase the UID should be consitient for a card, and the fraud flag should as well, the goal is to get the lowest amount of mixed values

In [ ]:
# create UIDs to test against

# uid_2 does not use the email address
df['uid_2'] = df['D1_new'].astype(str) + '_' + df['card1'].astype(str) + '_' + df['addr1'].astype(str)

# uid_3 uses uid_1 + addr2 (address continued but often null)
df['uid_3'] = df['D1_new'].astype(str) + '_' + df['P_emaildomain'].astype(str) + '_' + df['card1'].astype(str) + '_' + df['addr1'].astype(str) + '_' + df['addr2'].astype(str)

# uid_4 uses uid_1 + v313 (0 or first transaction amount for card)
df['uid_4'] = df['D1_new'].astype(str) + '_' + df['P_emaildomain'].astype(str) + '_' + df['card1'].astype(str) + '_' + df['addr1'].astype(str) + '_' + df['V313'].astype(str)

In [ ]:
import pandas as pd

# test UIDs against each other
uid_list = ['uid_1', 'uid_2', 'uid_3', 'uid_4']

results = []
columns = ['uid_X', 'Fraud Only UIDs', 'Normal Only UIDs', 'Mixed Fraud UIDs']

# make dataframe with the percentage of fraud==0 and fraud==1 values for each uid
df_results = pd.DataFrame(columns = columns)

i = 0

for uid in uid_list:
  # aggregate min/max of 'isFraud
  grouped_fraud = df.groupby(uid)['isFraud'].agg(['min', 'max'])

  fraud_only_uids_count = len(grouped_fraud[(grouped_fraud['min'] == 1) & (grouped_fraud['max'] == 1)])
  normal_only_uids_count = len(grouped_fraud[(grouped_fraud['min'] == 0) & (grouped_fraud['max'] == 0)])
  mixed_uids_count = len(grouped_fraud[(grouped_fraud['min'] == 0) & (grouped_fraud['max'] == 1)])

  results = [uid, fraud_only_uids_count, normal_only_uids_count, mixed_uids_count]
  df_results.loc[i] = results
  i += 1

# convert df_results numeric columns to percentages
df_results['Total UIDs'] = df_results['Fraud Only UIDs'] + df_results['Normal Only UIDs'] + df_results['Mixed Fraud UIDs']
df_results['Fraud Only UIDs'] = df_results['Fraud Only UIDs'] / df_results['Total UIDs'] * 100
df_results['Normal Only UIDs'] = df_results['Normal Only UIDs'] / df_results['Total UIDs'] * 100
df_results['Mixed Fraud UIDs'] = df_results['Mixed Fraud UIDs'] / df_results['Total UIDs'] * 100

display(df_results)

From this we can see that uid_2 definitely does not work as intended. The other UIDs have the same mixed fraud rates, while their 'Fraud Only' and 'Normal Only' generally have different amounts. uid_1 and uid_3 are in agreement, meaning that adding addr2 to our analysis did not improve for hurt the analysis.

That leaves deciding between uid_1 and uid_4. uid_4 is slightly better than uid_1.